<a href="https://colab.research.google.com/github/aodm26/efficientNet/blob/main/EfficientNetB0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# EfficientNetB0 – Cardiomegaly Detection (Transfer Learning)
# DS7023 Component 1
# ============================================================
import os, time
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, roc_curve
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

try:
    from google.colab import files, drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

PyTorch 2.10.0+cpu, CUDA: False
Using device: cpu


In [ ]:
# Cell 2: Upload & Extract the 4 ZIPs
# ---------------------------------------------------------------------------
# Upload: trainCardiomegaly.zip, trainNo_Finding.zip, validation.zip, test.zip
# ---------------------------------------------------------------------------

uploaded = files.upload()  # upload cardiomegaly_dataset.zip only
!unzip -q cardiomegaly_dataset.zip -d /content/
data_dir = '/content/data'

Saving cardiomegaly_dataset.zip to cardiomegaly_dataset.zip


In [ ]:
# Cell 3: Hyperparameters & Transforms
# ---------------------------------------------------------------------------
# Images are GRAYSCALE – replicated to 3 channels via Grayscale(3).
# Two-phase transfer learning:
#   Phase 1 (epochs 1-5):   freeze backbone, train head only  (LR = 1e-3)
#   Phase 2 (epochs 6-25):  unfreeze all, differential LR
#                           backbone = 1e-5, head = 1e-4
# ---------------------------------------------------------------------------

BATCH_SIZE    = 32
NUM_EPOCHS    = 25
FREEZE_EPOCHS = 5
LR_HEAD       = 1e-3
LR_BACKBONE   = 1e-5
WEIGHT_DECAY  = 1e-4
IMG_SIZE      = 224
SEED          = 42

torch.manual_seed(SEED); np.random.seed(SEED)

mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),  # grayscale -> 3-ch
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

eval_tfms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_ds = datasets.ImageFolder(os.path.join(data_dir, 'train'),      transform=train_tfms)
val_ds   = datasets.ImageFolder(os.path.join(data_dir, 'validation'), transform=eval_tfms)
test_ds  = datasets.ImageFolder(os.path.join(data_dir, 'test'),       transform=eval_tfms)

print(f'Classes : {train_ds.classes}  ->  {train_ds.class_to_idx}')
print(f'Train {len(train_ds)} | Val {len(val_ds)} | Test {len(test_ds)}')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

Classes : ['Cardiomegaly', 'No Finding']  ->  {'Cardiomegaly': 0, 'No Finding': 1}
Train 1600 | Val 384 | Test 200


In [ ]:
# Cell 4: Build EfficientNetB0 Model
def build_model(num_classes=2):
    weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
    m = models.efficientnet_b0(weights=weights)
    # Replace classifier head
    in_f = m.classifier[1].in_features  # 1280
    m.classifier = nn.Sequential(
        nn.Dropout(p=0.4, inplace=True),
        nn.Linear(in_f, 256), nn.ReLU(inplace=True),
        nn.Dropout(p=0.3),
        nn.Linear(256, num_classes))
    return m

model = build_model().to(device)

# Phase 1: freeze backbone
def set_backbone_grad(m, requires_grad):
    for name, p in m.named_parameters():
        if 'classifier' not in name:
            p.requires_grad = requires_grad

set_backbone_grad(model, False)
total      = sum(p.numel() for p in model.parameters())
trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total:,} | Trainable (Phase 1): {trainable:,}')

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 97.9MB/s]


Total params: 4,335,998 | Trainable (Phase 1): 328,450


In [ ]:
# Cell 5: Training Loop (2-Phase)
criterion  = nn.CrossEntropyLoss()
optimizer  = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
MODEL_PATH = 'efficientnet_b0_best.pt'

def run_epoch(model, loader, optimizer=None, criterion=None, training=True):
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            if training: optimizer.zero_grad()
            outs = model(imgs)
            loss = criterion(outs, lbls)
            if training: loss.backward(); optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            probs = F.softmax(outs,1)[:,1].detach().cpu().numpy()
            all_probs.extend(probs); all_labels.extend(lbls.cpu().numpy())
            correct += (outs.argmax(1)==lbls).sum().item(); total += lbls.size(0)
    auc = roc_auc_score(all_labels, all_probs) if len(set(all_labels))>1 else 0.0
    return total_loss/total, correct/total, auc, all_labels, all_probs


history = {k:[] for k in ['train_loss','train_acc','val_loss','val_acc','val_auc']}
best_auc = best_epoch = 0
print(f'Training {NUM_EPOCHS} epochs on {device}...\n')
t0 = time.time()

for ep in range(1, NUM_EPOCHS+1):

    # Switch to Phase 2
    if ep == FREEZE_EPOCHS + 1:
        print(f'\n--- Phase 2: Unfreezing backbone at epoch {ep} ---')
        set_backbone_grad(model, True)
        optimizer = torch.optim.Adam([
            {'params': model.features.parameters(),   'lr': LR_BACKBONE},
            {'params': model.classifier.parameters(), 'lr': LR_HEAD/10},
        ], weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=NUM_EPOCHS-FREEZE_EPOCHS)
        print(f'Trainable params (Phase 2): {sum(p.numel() for p in model.parameters() if p.requires_grad):,}\n')

    tl,ta,_,_,_ = run_epoch(model, train_loader, optimizer, criterion, True)
    vl,va,va_auc,_,_ = run_epoch(model, val_loader, None, criterion, False)
    scheduler.step()

    for k,v in zip(['train_loss','train_acc','val_loss','val_acc','val_auc'],[tl,ta,vl,va,va_auc]):
        history[k].append(v)
    if va_auc > best_auc:
        best_auc,best_epoch = va_auc,ep; torch.save(model.state_dict(), MODEL_PATH)

    phase = 'P1' if ep<=FREEZE_EPOCHS else 'P2'
    print(f'[{phase}] Ep {ep:02d}/{NUM_EPOCHS} | Train Loss {tl:.4f} Acc {ta:.4f} | Val Loss {vl:.4f} Acc {va:.4f} AUC {va_auc:.4f}' + (' *** BEST' if ep==best_epoch else ''))

print(f'\nDone in {(time.time()-t0)/60:.1f} min | Best Val AUC: {best_auc:.4f} @ ep {best_epoch}')

Training 25 epochs on cpu...



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[P1] Ep 01/25 | Train Loss 0.6639 Acc 0.5950 | Val Loss 0.6353 Acc 0.6380 AUC 0.7031 *** BEST


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[P1] Ep 02/25 | Train Loss 0.6133 Acc 0.6637 | Val Loss 0.6061 Acc 0.6693 AUC 0.7334 *** BEST


...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[P2] Ep 20/25 | Train Loss 0.4556 Acc 0.7831 | Val Loss 0.5041 Acc 0.7682 AUC 0.8393 *** BEST


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[P2] Ep 21/25 | Train Loss 0.4408 Acc 0.8025 | Val Loss 0.5038 Acc 0.7682 AUC 0.8411 *** BEST


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
# Cell 6: Training Curves
eps = range(1, NUM_EPOCHS+1)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(eps,history['train_loss'],label='Train'); ax[0].plot(eps,history['val_loss'],label='Val')
ax[0].axvline(FREEZE_EPOCHS,color='gray',linestyle=':',label='Unfreeze')
ax[0].set_title('Loss – EfficientNetB0'); ax[0].legend(); ax[0].set_xlabel('Epoch')
ax[1].plot(eps,history['train_acc'],label='Train'); ax[1].plot(eps,history['val_acc'],label='Val')
ax[1].axvline(FREEZE_EPOCHS,color='gray',linestyle=':')
ax[1].set_title('Accuracy – EfficientNetB0'); ax[1].legend(); ax[1].set_xlabel('Epoch')
ax[2].plot(eps,history['val_auc'],color='green')
ax[2].axvline(best_epoch,color='red',linestyle='--',label=f'Best @ ep{best_epoch}')
ax[2].axvline(FREEZE_EPOCHS,color='gray',linestyle=':',label='Unfreeze')
ax[2].set_title('Val AUC-ROC – EfficientNetB0'); ax[2].legend(); ax[2].set_xlabel('Epoch')
plt.suptitle('EfficientNetB0 – Training History', fontsize=13)
plt.tight_layout(); plt.savefig('efficientnet_training_curves.png',dpi=150); plt.show()

In [ ]:
# Cell 7: Test Evaluation
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
_,test_acc,test_auc,y_true,y_prob = run_epoch(model, test_loader, None, criterion, False)
y_pred = [1 if p>=0.5 else 0 for p in y_prob]

print('='*55, '\nTEST RESULTS – EfficientNetB0\n' + '='*55)
print(f'Accuracy : {test_acc:.4f}\nAUC-ROC  : {test_auc:.4f}\n')
print(classification_report(y_true, y_pred, target_names=train_ds.classes))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=train_ds.classes, yticklabels=train_ds.classes, ax=axes[0])
axes[0].set_title('Confusion Matrix – EfficientNetB0')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')
fpr,tpr,_ = roc_curve(y_true, y_prob)
axes[1].plot(fpr,tpr,color='green',lw=2,label=f'AUC={test_auc:.4f}'); axes[1].plot([0,1],[0,1],'k--')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve – EfficientNetB0'); axes[1].legend()
plt.tight_layout(); plt.savefig('efficientnet_eval.png',dpi=150); plt.show()

In [ ]:
# Cell 8: Grad-CAM Explainability
class GradCAM:
    def __init__(self, model, layer):
        self.grads = self.acts = None
        layer.register_forward_hook(lambda m,i,o: setattr(self,'acts',o.detach()))
        layer.register_backward_hook(lambda m,gi,go: setattr(self,'grads',go[0].detach()))
        self.model = model

    def __call__(self, x, cls=None):
        self.model.eval()
        out = self.model(x)
        if cls is None: cls = out.argmax(1).item()
        self.model.zero_grad(); out[0,cls].backward()
        w = self.grads.mean(dim=[2,3], keepdim=True)
        cam = F.relu((w * self.acts).sum(1, keepdim=True))
        cam = cam - cam.min(); cam = cam / (cam.max() + 1e-8)
        return cam.squeeze().cpu().numpy(), cls

gradcam = GradCAM(model, model.features[-1][0])
inv = transforms.Normalize([-m/s for m,s in zip(mean,std)],[1/s for s in std])
imgs_b, lbls_b = next(iter(test_loader))

fig, axs = plt.subplots(2, 5, figsize=(18, 7))
for i in range(5):
    img_t = imgs_b[i:i+1].to(device)
    cam, pred = gradcam(img_t)
    img_np = inv(imgs_b[i]).permute(1,2,0).clamp(0,1).numpy()
    cam_up = np.array(Image.fromarray((cam*255).astype('uint8')).resize((224,224)))
    axs[0,i].imshow(img_np[:,:,0], cmap='gray')
    axs[0,i].set_title(f'True: {train_ds.classes[lbls_b[i]]}', fontsize=8); axs[0,i].axis('off')
    axs[1,i].imshow(img_np[:,:,0], cmap='gray')
    axs[1,i].imshow(cam_up, cmap='jet', alpha=0.45)
    axs[1,i].set_title(f'Pred: {train_ds.classes[pred]}', fontsize=8); axs[1,i].axis('off')

plt.suptitle('Grad-CAM – EfficientNetB0 (row 1: original, row 2: activation overlay)', fontsize=12)
plt.tight_layout(); plt.savefig('gradcam_visualisation.png',dpi=150); plt.show()

In [ ]:
# Cell 9: Download outputs
if IN_COLAB:
    for f in ['efficientnet_b0_best.pt','efficientnet_training_curves.png',
              'efficientnet_eval.png','gradcam_visualisation.png']:
        files.download(f)
print('All done.')